<a href="https://colab.research.google.com/github/priyal6/DL/blob/main/vgg_pre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = datasets.CIFAR10(root = './data', train = True, download = True, transform = transform)
test_data = datasets.CIFAR10(root='./data', train = False, download=False, transform=transform)

100%|██████████| 170M/170M [00:14<00:00, 11.5MB/s]


In [3]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2 )

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vgg = models.vgg16(weights = 'IMAGENET1K_V1')

for param in vgg.features.parameters():
  param.requires_grad=False

vgg.classifier[6] = nn.Linear(4096, 10)

vgg = vgg.to(device)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 166MB/s]


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg.classifier.parameters(), lr=0.001)

for epoch in range(3):  # start small
    vgg.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = vgg(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")


Epoch 1, Loss: 1.0940
Epoch 2, Loss: 0.8452
Epoch 3, Loss: 0.7504


In [6]:
vgg.eval()
correct, total = 0,0
with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = vgg(images)
    _, predicted = torch.max(outputs,1)
    total = total + labels.size(0)
    correct += (predicted == labels).sum().item()
print(f"Test Accuracy: {100*correct/total:.2f}%")



Test Accuracy: 78.92%
